# 📱 PhoneTrackPredictor

**Reconhecendo atividades humanas a partir dos sensores do celular, usando Árvores de Decisão.**

Os sensores de um smartphone (acelerômetro e giroscópio) geram sinais diferentes quando a pessoa está caminhando, sentada ou deitada. Neste projeto eu treino uma árvore de decisão para identificar qual das 6 atividades está acontecendo, e depois tento deixá-la melhor escolhendo com cuidado quais variáveis usar.

| Código | Atividade |
|:--:|:--|
| 1 | Caminhando |
| 2 | Subindo escada |
| 3 | Descendo escada |
| 4 | Sentado |
| 5 | Em pé |
| 6 | Deitado |

**O que vem pela frente**

1. Preparação do ambiente
2. Carregando os dados
3. Primeira árvore (com poda por *cost complexity*)
4. Avaliação no conjunto de teste
5. Encontrando variáveis melhores
6. Árvore melhorada e comparação final
7. Conclusões

## 1. Preparação do ambiente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

# Configurações do projeto, concentradas aqui para facilitar ajustes
RANDOM_STATE = 42
MIN_SAMPLES_LEAF = 20   # folhas com pelo menos 20 amostras, para evitar árvores "decoradas"
K_FOLDS = 10            # número de dobras na validação cruzada

ATIVIDADES = {
    1: "Caminhando",
    2: "Subindo escada",
    3: "Descendo escada",
    4: "Sentado",
    5: "Em pé",
    6: "Deitado",
}

# As 3 variáveis que serviram de ponto de partida
VARIAVEIS_BASE = [
    "tGravityAcc-energy()-X",
    "fBodyAccJerk-bandsEnergy()-1,8",
    "angle(Y,gravityMean)",
]

## 2. Carregando os dados

Uso as bases de treino e teste exatamente como já vieram separadas originalmente. As colunas das bases de treino e teste são as mesmas, e os rótulos (`y`) ficam em arquivos de texto com uma atividade por linha, sem cabeçalho.

In [ ]:
X_treino = pd.read_csv("X_train_ajustado.csv")
X_teste = pd.read_csv("X_teste_ajustado.csv")

# Os arquivos de rótulos não têm cabeçalho, então header=None evita perder a primeira linha
y_treino = pd.read_csv("y_train.txt", header=None).squeeze("columns")
y_teste = pd.read_csv("y_test.txt", header=None).squeeze("columns")

# Checagem de sanidade: cada linha de X precisa ter um rótulo correspondente
assert len(X_treino) == len(y_treino), f"Treino desalinhado: X={len(X_treino)} / y={len(y_treino)}"
assert len(X_teste) == len(y_teste), f"Teste desalinhado: X={len(X_teste)} / y={len(y_teste)}"

print(f"Treino: {X_treino.shape[0]} amostras, {X_treino.shape[1]} variáveis")
print(f"Teste:  {X_teste.shape[0]} amostras, {X_teste.shape[1]} variáveis")

## 3. Primeira árvore: encontrando o nível certo de poda

Uma árvore sem limites tende a decorar os dados de treino (*overfitting*). A poda por *cost complexity* resolve isso com o parâmetro `ccp_alpha`: quanto maior o valor, mais a árvore é simplificada.

Mas qual valor usar? Em vez de chutar, deixo o `GridSearchCV` testar todos os valores possíveis com validação cruzada (*k-fold*, k = 10) e escolher o melhor. Para começar, uso apenas as 3 variáveis base.

Vou criar uma função para esse processo, porque vou reaproveitá-la mais adiante.

In [ ]:
def buscar_melhor_arvore(X, y):
    """Testa todos os ccp_alpha possíveis com validação cruzada e devolve a busca já treinada."""
    arvore = DecisionTreeClassifier(random_state=RANDOM_STATE, min_samples_leaf=MIN_SAMPLES_LEAF)
    alphas = arvore.cost_complexity_pruning_path(X, y).ccp_alphas

    busca = GridSearchCV(arvore, param_grid={"ccp_alpha": alphas}, cv=K_FOLDS, n_jobs=-1)
    busca.fit(X, y)
    print(f"{len(alphas)} valores de ccp_alpha testados")
    print(f"Melhor ccp_alpha: {busca.best_params_['ccp_alpha']:.6f}")
    print(f"Acurácia média na validação cruzada: {busca.best_score_:.2%}")
    return busca

In [ ]:
%%time
busca_base = buscar_melhor_arvore(X_treino[VARIAVEIS_BASE], y_treino)

### Como a acurácia muda com a poda

O gráfico mostra o que a busca encontrou: com pouca poda a árvore é complexa demais, com poda excessiva ela fica simples demais, e o melhor resultado costuma estar no meio.

In [ ]:
resultados = pd.DataFrame(busca_base.cv_results_).sort_values("param_ccp_alpha")

plt.figure(figsize=(8, 4))
plt.plot(resultados["param_ccp_alpha"].astype(float), resultados["mean_test_score"], marker="o", markersize=4)
plt.axvline(busca_base.best_params_["ccp_alpha"], color="red", linestyle="--", label="Melhor ccp_alpha")
plt.xscale("symlog", linthresh=1e-4)
plt.xlabel("ccp_alpha (escala logarítmica)")
plt.ylabel("Acurácia média (validação cruzada)")
plt.title("Efeito da poda na acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Avaliação no conjunto de teste

Agora é a hora da verdade: a melhor árvore (já treinada com toda a base de treino) é avaliada em dados que ela nunca viu. Além da acurácia, a matriz de confusão mostra *onde* ela erra.

In [ ]:
def avaliar(modelo, X, y, titulo):
    """Calcula a acurácia no teste e desenha a matriz de confusão."""
    previsoes = modelo.predict(X)
    acuracia = accuracy_score(y, previsoes)

    ConfusionMatrixDisplay.from_predictions(
        y, previsoes,
        display_labels=[ATIVIDADES[c] for c in modelo.classes_],
        xticks_rotation=45, cmap="Blues", colorbar=False,
    )
    plt.title(f"{titulo}\nAcurácia no teste: {acuracia:.2%}")
    plt.tight_layout()
    plt.show()
    return acuracia, previsoes

In [ ]:
arvore_base = busca_base.best_estimator_
acuracia_base, previsoes_base = avaliar(arvore_base, X_teste[VARIAVEIS_BASE], y_teste, "Árvore base (3 variáveis)")

### Quais atividades a árvore mais confunde?

Nem toda atividade é igualmente difícil. Atividades parecidas (como sentado e em pé, ou os diferentes tipos de caminhada) tendem a se misturar. Vou medir o F1-score de cada classe para descobrir qual delas dá mais trabalho.

In [ ]:
f1_por_classe = f1_score(y_teste, previsoes_base, labels=arvore_base.classes_, average=None)

tabela_f1 = pd.DataFrame({
    "Atividade": [ATIVIDADES[c] for c in arvore_base.classes_],
    "F1-score": f1_por_classe,
}).sort_values("F1-score").reset_index(drop=True)

classe_mais_dificil = arvore_base.classes_[np.argmin(f1_por_classe)]
print(f"Atividade mais difícil de identificar: {ATIVIDADES[classe_mais_dificil]} (código {classe_mais_dificil})\n")
tabela_f1.style.format({"F1-score": "{:.2%}"}).hide(axis="index")

## 5. Buscando variáveis melhores

A forma mais eficiente de melhorar um modelo é dar a ele variáveis que realmente agreguem informação. Em vez de testar colunas no escuro (a base tem centenas delas), faço uma seleção guiada:

1. Transformo o problema em algo mais simples: **"é a atividade mais difícil ou não?"** (variável binária).
2. Treino uma árvore rasa (`max_depth=4`) para responder isso, usando **todas** as variáveis.
3. Olho a importância que ela deu a cada variável e fico com as 3 mais relevantes.

In [ ]:
y_treino_binario = (y_treino == classe_mais_dificil).astype(int)

arvore_selecao = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=RANDOM_STATE
).fit(X_treino, y_treino_binario)

importancias = (
    pd.Series(arvore_selecao.feature_importances_, index=X_treino.columns)
    .sort_values(ascending=False)
)
novas_variaveis = importancias.head(3).index.tolist()

importancias.head(10).sort_values().plot.barh(figsize=(8, 4), color="steelblue")
plt.title(f"Variáveis mais importantes para identificar: {ATIVIDADES[classe_mais_dificil]}")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

print("Variáveis escolhidas:")
for v in novas_variaveis:
    print(f"  • {v}")

## 6. Árvore melhorada

Junto as 3 variáveis novas com as 3 base e repito todo o processo: busca do melhor `ccp_alpha` e avaliação no teste.

In [ ]:
variaveis_melhoradas = VARIAVEIS_BASE + [v for v in novas_variaveis if v not in VARIAVEIS_BASE]
print(f"Total de variáveis: {len(variaveis_melhoradas)}")

In [ ]:
%%time
busca_melhorada = buscar_melhor_arvore(X_treino[variaveis_melhoradas], y_treino)

In [ ]:
arvore_melhorada = busca_melhorada.best_estimator_
acuracia_melhorada, _ = avaliar(arvore_melhorada, X_teste[variaveis_melhoradas], y_teste, f"Árvore melhorada ({len(variaveis_melhoradas)} variáveis)")

### Comparação final

In [ ]:
comparacao = pd.DataFrame({
    "Modelo": ["Árvore base", "Árvore melhorada"],
    "Variáveis": [len(VARIAVEIS_BASE), len(variaveis_melhoradas)],
    "Acurácia no teste": [acuracia_base, acuracia_melhorada],
})
ganho = acuracia_melhorada - acuracia_base
print(f"Ganho de acurácia: {ganho:+.2%} (pontos percentuais)\n")
comparacao.style.format({"Acurácia no teste": "{:.2%}"}).hide(axis="index")

## 7. Conclusões

- **Poucas variáveis já levam longe.** Só com 3 sensores bem escolhidos, uma árvore simples consegue separar bem as atividades.
- **Os erros não são aleatórios.** Eles se concentram em atividades parecidas entre si, e a matriz de confusão mostra exatamente onde.
- **Selecionar variáveis com método funciona melhor do que chutar.** Usar a árvore binária para ranquear importâncias é um caminho barato e direto para melhorar o modelo.
- **A poda ajuda a generalizar.** O `ccp_alpha` escolhido por validação cruzada evita que a árvore decore o treino.

**Ideias para os próximos passos**

- Testar outros modelos, como Random Forest e Gradient Boosting, e comparar com a árvore simples.
- Repetir a seleção de variáveis para as outras classes difíceis, não só para uma.
- Usar uma seleção automática (por exemplo, `SelectFromModel` ou `RFE`) e comparar com a abordagem manual.